# RT-DETR 실제 INT8 / INT4 PTQ 평가
COCO 사전학습 RT-DETR을 FP32 ONNX로 내보내고, 같은 모델에서 **정적 INT8 PTQ**와 **packed INT4 weight-only PTQ** 모델을 생성합니다. 세 모델을 같은 COCO val2017 평가 1,000장과 ONNX Runtime CPU에서 비교합니다. CPU 런타임으로 실행하면 GPU 배정 여부가 결과에 영향을 주지 않습니다.

INT8 대상은 Conv·MatMul의 가중치와 활성값입니다. INT4 대상은 상수 가중치를 가진 MatMul뿐이며, Conv와 활성값은 FP32로 남습니다. **INT4 결과는 전체 RT-DETR W4A4가 아닙니다.** 노트북은 실제 양자화된 `.onnx` 파일과 런타임 정수 연산 수를 확인합니다.


In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/content/haiteam4_finetune')
if repo.exists():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/foxyaef/haiteam4_finetune.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt', '-r', 'requirements-onnx.txt'], check=True)
import torch, onnxruntime
print('PyTorch:', torch.__version__, 'ONNX Runtime:', onnxruntime.__version__)


In [ ]:
subprocess.run([sys.executable, 'scripts/colab_setup.py'], check=True)
subprocess.run([sys.executable, 'prepare_data.py'], check=True)
subprocess.run([sys.executable, 'prepare_data.py', '--verify-only'], check=True)


## 실제 모델 생성
FP32 ONNX export를 한 뒤 COCO calibration 512장으로 INT8 정적 PTQ를 수행합니다. INT4는 상수 가중치 MatMul을 4비트로 패킹하는 weight-only PTQ입니다. 각각 독립적인 ONNX 파일로 저장됩니다.


In [ ]:
fp32 = repo / 'artifacts/rtdetr_fp32.onnx'
int8 = repo / 'artifacts/rtdetr_int8.onnx'
int4 = repo / 'artifacts/rtdetr_int4.onnx'
if not fp32.exists():
    subprocess.run([sys.executable, 'scripts/onnx_real_ptq.py', 'export', '--output', str(fp32)], check=True)
print('FP32:', fp32, fp32.stat().st_size / 2**20, 'MiB')


이미 양자화한 `.onnx` 파일 하나만 재평가하려면 바로 아래 셀의 `uploaded_precision`을 `int8` 또는 `int4`로 바꾸고 업로드하세요. 기본값 `None`은 코랩에서 둘 다 생성합니다.


In [ ]:
uploaded_precision = None  # None, 'int8', 'int4' 중 선택
if uploaded_precision is not None:
    if uploaded_precision not in ('int8', 'int4'):
        raise ValueError('int8 또는 int4를 선택하세요')
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1 or not next(iter(uploaded)).endswith('.onnx'):
        raise ValueError('.onnx 파일 하나를 올리세요')
    uploaded_path = pathlib.Path(next(iter(uploaded))).resolve()
    if uploaded_precision == 'int8':
        int8 = uploaded_path
    else:
        int4 = uploaded_path
    del uploaded


In [ ]:
if uploaded_precision in (None, 'int8'):
    if not int8.exists():
        subprocess.run([sys.executable, 'scripts/onnx_real_ptq.py', 'quantize', '--source', str(fp32), '--precision', 'int8', '--output', str(int8)], check=True)
    print('INT8:', int8, int8.stat().st_size / 2**20, 'MiB')


In [ ]:
if uploaded_precision in (None, 'int4'):
    if not int4.exists():
        subprocess.run([sys.executable, 'scripts/onnx_real_ptq.py', 'quantize', '--source', str(fp32), '--precision', 'int4', '--output', str(int4)], check=True)
    print('INT4:', int4, int4.stat().st_size / 2**20, 'MiB')


## 정확도·시간·메모리 평가
각 모델을 별도 프로세스에서 평가해 메모리 피크를 분리합니다. 최초 실행은 CPU에서 오래 걸릴 수 있습니다. 모델 내부 메타데이터로 동일한 COCO 기준인지 확인합니다.


In [ ]:
import datetime
tag = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d_%H%M%S')
selected = ['int8', 'int4'] if uploaded_precision is None else [uploaded_precision]
models = [('fp32', fp32)] + [(kind, {'int8': int8, 'int4': int4}[kind]) for kind in selected]
run_ids = {kind: 'onnx_' + kind + '_' + tag for kind, _ in models}
for kind, model in models:
    subprocess.run([sys.executable, 'scripts/onnx_eval.py', '--model', str(model), '--run-id', run_ids[kind]], check=True)
print('Completed:', run_ids)


In [ ]:
from google.colab import files
import zipfile
bundle = repo / ('real_ptq_results_' + tag + '.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zipped:
    for run_id in run_ids.values():
        for name in ('metrics.json', 'class_metrics.csv', 'profile.json', 'provenance.json'):
            result = repo / 'runs/onnx_eval' / run_id / name
            zipped.write(result, arcname=run_id + '/' + name)
files.download(str(bundle))
print('로컬 분석: python scripts/analyze_real_ptq.py', bundle.name)


## 모델 파일 다운로드 (선택)
실제 양자화된 모델을 보관하려면 아래 셀에서 원하는 파일을 내려받으세요. `.onnx` **파일 하나만으로** 재평가할 수 있도록 모델 정보가 파일 내부에 들어 있습니다.


In [ ]:
# 아래 중 원하는 모델을 선택하세요: fp32, int8, int4
model_to_download = int4
files.download(str(model_to_download))
